In [1]:

%%writefile generate_data.py

import numpy as np
import pandas as pd

# ==========================================
# GENERATOR 1: DUKA ELECTRONICS
# =========================S=================

rng = np.random.default_rng(2026)

n = 1000

products = {
    "Phone": 15000,
    "Laptop": 65000,
    "Headphones": 2500,
    "Tablet": 28000,
    "Charger": 1200
}

sales = pd.DataFrame({
    "order_id": range(1001, 1001 + n),

    "order_date": pd.to_datetime("2026-01-01")
                  + pd.to_timedelta(rng.integers(0, 180, n), unit="D"),

    "branch": rng.choice(
        ["Nairobi", "Mombasa", "Kisumu", "Nakuru", "Eldoret"],
        n,
        p=[0.4, 0.2, 0.15, 0.15, 0.1]
    ),

    "product": rng.choice(list(products), n),

    "quantity": rng.integers(1, 6, n),

    "channel": rng.choice(
        ["In-store", "Online", "Phone order"],
        n,
        p=[0.55, 0.35, 0.10]
    ),
})

sales["unit_price"] = sales["product"].map(products)

sales.to_csv("duka_sales.csv", index=False)


# ==========================================
# GENERATOR 2: MOBILE MONEY AGENT NETWORK
# ==========================================

rng = np.random.default_rng(44)

n = 600

agents = pd.DataFrame({
    "agent_id": [f"AG{i:03d}" for i in range(1, 21)],

    "county": rng.choice(
        ["Nairobi", "Kiambu", "Nakuru", "Mombasa", "Kisumu"],
        20
    ),

    "onboarded": pd.date_range(
        "2024-01-01",
        periods=20,
        freq="MS"
    ).strftime("%Y-%m-%d"),
})

txns = pd.DataFrame({
    "txn_id": [f"TX{i:05d}" for i in range(1, n + 1)],

    "agent_id": rng.choice(agents["agent_id"], n),

    "txn_type": rng.choice(
        ["Deposit", "withdrawal", "DEPOSIT ", "Withdrawal", "deposit"],
        n
    ),

    "amount": rng.lognormal(7.5, 0.8, n).round(0).astype(str),

    "txn_date": (
        pd.to_datetime("2026-03-01")
        + pd.to_timedelta(rng.integers(0, 60, n), unit="D")
    ).strftime("%Y-%m-%d"),
})

idx = rng.choice(n, 30, replace=False)

txns.loc[idx[:15], "amount"] = np.nan
txns.loc[idx[15:20], "amount"] = "N/A"
txns.loc[idx[20:25], "amount"] = "250000"
txns.loc[idx[25:], "txn_date"] = (
    txns.loc[idx[25:], "txn_date"].str.replace("-", "/")
)

txns = pd.concat(
    [txns, txns.sample(12, random_state=1)],
    ignore_index=True
)

txns.loc[
    rng.choice(len(txns), 8, replace=False),
    "agent_id"
] = "AG099"

agents.to_csv("agents.csv", index=False)

txns.to_csv("agent_transactions.csv", index=False)

print("All three datasets generated successfully!")


Writing generate_data.py


In [2]:
%run generate_data.py

All three datasets generated successfully!


In [3]:
import os

print(os.listdir())

['.config', 'agent_transactions.csv', 'generate_data.py', 'duka_sales.csv', 'agents.csv', 'sample_data']


In [4]:
import pandas as pd
import numpy as np

In [5]:
sales = pd.read_csv(
    "duka_sales.csv",
    parse_dates=["order_date"]
)

In [6]:
sales.head()

,order_id,order_date,branch,product,quantity,channel,unit_price
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500


In [7]:
sales.shape

(1000, 7)

In [8]:
sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   order_id    1000 non-null   int64         
 1   order_date  1000 non-null   datetime64[ns]
 2   branch      1000 non-null   object        
 3   product     1000 non-null   object        
 4   quantity    1000 non-null   int64         
 5   channel     1000 non-null   object        
 6   unit_price  1000 non-null   int64         
dtypes: datetime64[ns](1), int64(3), object(3)
memory usage: 54.8+ KB


In [9]:
sales.isnull().sum()

,0
order_id,0
order_date,0
branch,0
product,0
quantity,0
channel,0
unit_price,0


In [10]:
print("Earliest date:", sales["order_date"].min())
print("Latest date:", sales["order_date"].max())


Earliest date: 2026-01-01 00:00:00
Latest date: 2026-06-29 00:00:00


In [11]:
sales["revenue"] = sales["quantity"] * sales["unit_price"]

sales.head()

,order_id,order_date,branch,product,quantity,channel,unit_price,revenue
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000,45000
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000,260000
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000,140000
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500,12500
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500,7500


In [12]:
sales["month"] = sales["order_date"].dt.month_name()

sales.head()

,order_id,order_date,branch,product,quantity,channel,unit_price,revenue,month
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000,45000,June
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000,260000,February
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000,140000,January
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500,12500,April
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500,7500,March


In [13]:
sales["weekday"] = sales["order_date"].dt.day_name()

sales.head()

,order_id,order_date,branch,product,quantity,channel,unit_price,revenue,month,weekday
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000,45000,June,Wednesday
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000,260000,February,Monday
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000,140000,January,Monday
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500,12500,April,Sunday
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500,7500,March,Saturday


In [14]:
sales["is_weekend"] = sales["weekday"].isin(["Saturday", "Sunday"])

sales.head()

,order_id,order_date,branch,product,quantity,channel,unit_price,revenue,month,weekday,is_weekend
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000,45000,June,Wednesday,False
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000,260000,February,Monday,False
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000,140000,January,Monday,False
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500,12500,April,Sunday,True
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500,7500,March,Saturday,True


In [15]:
sales["price_band"] = np.where(
    sales["unit_price"] >= 28000,
    "Premium",
    "Value"
)

sales.head()

,order_id,order_date,branch,product,quantity,channel,unit_price,revenue,month,weekday,is_weekend,price_band
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000,45000,June,Wednesday,False,Value
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000,260000,February,Monday,False,Premium
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000,140000,January,Monday,False,Premium
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500,12500,April,Sunday,True,Value
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500,7500,March,Saturday,True,Value


In [16]:
branch_revenue = sales.groupby("branch")["revenue"].sum()

branch_revenue.sort_values(ascending=False)

,revenue
branch,
Nairobi,26106100
Mombasa,14003700
Nakuru,11970400
Kisumu,10868100
Eldoret,8844200


In [17]:
branch_orders = sales.groupby("branch")["order_id"].count()

branch_orders.sort_values(ascending=False)

,order_id
branch,
Nairobi,376
Mombasa,186
Nakuru,162
Kisumu,159
Eldoret,117


In [19]:
branch_comparison = pd.DataFrame({
    "total_revenue": branch_revenue,
    "total_orders": branch_orders
})

branch_comparison.sort_values(
    by="total_revenue",
    ascending=False
)

,total_revenue,total_orders
branch,,
Nairobi,26106100,376
Mombasa,14003700,186
Nakuru,11970400,162
Kisumu,10868100,159
Eldoret,8844200,117


### Task 3: Branch Performance

Nairobi was the best-performing branch, generating KES 26,106,100 from 376 orders. Eldoret had the lowest performance, generating KES 8,844,200 from 117 orders. The ranking by revenue was the same as the ranking by order count. This suggests that branches receiving more orders generally generated more revenue in this dataset.

In [20]:
product_revenue = sales.groupby("product")["revenue"].sum()

product_revenue.sort_values(ascending=False)

,revenue
product,
Laptop,41535000
Tablet,19348000
Phone,8880000
Headphones,1387500
Charger,642000


In [23]:
product_units = sales.groupby("product")["quantity"].sum()

product_units.sort_values(ascending=False)

,quantity
product,
Tablet,691
Laptop,639
Phone,592
Headphones,555
Charger,535


### Task 4: Product Performance

Laptops generated the highest revenue at KES 41,535,000, while tablets had the highest number of units sold at 691. This difference occurs because products have different selling prices. Although more tablets were sold, laptops generated more revenue because their unit price is higher. Chargers generated the lowest revenue at KES 642,000.

In [25]:
sales.groupby("is_weekend")["order_id"].count()

,order_id
is_weekend,
False,723
True,277


In [26]:
sales.groupby("is_weekend")["revenue"].sum()

,revenue
is_weekend,
False,51856200
True,19936300


In [27]:
sales.groupby("is_weekend")["revenue"].mean()

,revenue
is_weekend,
False,71723.651452
True,71972.202166


### Task 5: Weekend vs Weekday Performance

Weekdays recorded 723 orders and generated KES 51,856,200 in revenue, while weekends recorded 277 orders and generated KES 19,936,300. However, the average revenue per order was slightly higher on weekends at KES 71,972.20 compared with KES 71,723.65 on weekdays. This shows that weekdays generated more total revenue because they had more orders, while weekend orders had a slightly higher average value.

In [28]:
monthly_revenue = sales.groupby("month")["revenue"].sum()

monthly_revenue

,revenue
month,
April,13518300
February,11772000
January,11786600
June,10903400
March,12396800
May,11415400


In [29]:
best_month_revenue = monthly_revenue.max()
worst_month_revenue = monthly_revenue.min()

percentage_difference = (
    (best_month_revenue - worst_month_revenue)
    / worst_month_revenue
) * 100

print("Best month revenue:", best_month_revenue)
print("Worst month revenue:", worst_month_revenue)
print("Percentage difference:", round(percentage_difference, 2), "%")

Best month revenue: 13518300
Worst month revenue: 10903400
Percentage difference: 23.98 %


### Task 6: Monthly Sales Performance

April was the best-performing month, generating KES 13,518,300 in revenue. June was the worst-performing month, generating KES 10,903,400. April's revenue was 23.98% higher than June's revenue. This suggests that sales performance was stronger in April than in June.

In [30]:
premium_online = sales[
    (sales["price_band"] == "Premium") &
    (sales["channel"] == "Online") &
    (sales["branch"].isin(["Nairobi", "Mombasa"]))
]

premium_online.head()

,order_id,order_date,branch,product,quantity,channel,unit_price,revenue,month,weekday,is_weekend,price_band
6,1007,2026-01-15,Nairobi,Laptop,2,Online,65000,130000,January,Thursday,False,Premium
13,1014,2026-06-12,Nairobi,Tablet,3,Online,28000,84000,June,Friday,False,Premium
32,1033,2026-04-29,Nairobi,Laptop,2,Online,65000,130000,April,Wednesday,False,Premium
35,1036,2026-03-02,Nairobi,Laptop,4,Online,65000,260000,March,Monday,False,Premium
39,1040,2026-02-10,Mombasa,Laptop,5,Online,65000,325000,February,Tuesday,False,Premium


In [31]:
print("Number of orders:", len(premium_online))

print("Total revenue:", premium_online["revenue"].sum())

Number of orders: 83
Total revenue: 10218000


### Task 7: Premium Online Sales

A total of 83 orders met the filtering conditions: Premium products purchased through the Online channel at the Nairobi or Mombasa branches. These orders generated total revenue of KES 10,218,000.

In [32]:
import os

os.makedirs("outputs", exist_ok=True)

premium_online.to_csv(
    "outputs/premium_online_coast_capital.csv",
    index=False
)

print("CSV file exported successfully!")

CSV file exported successfully!


In [33]:
branch_summary = sales.groupby("branch").agg(
    total_revenue=("revenue", "sum"),
    total_orders=("order_id", "count")
).reset_index()

branch_summary

,branch,total_revenue,total_orders
0,Eldoret,8844200,117
1,Kisumu,10868100,159
2,Mombasa,14003700,186
3,Nairobi,26106100,376
4,Nakuru,11970400,162


In [34]:
region_lookup = pd.DataFrame({
    "branch": ["Nairobi", "Mombasa", "Kisumu", "Nakuru", "Eldoret"],
    "region": ["Central", "Coast", "Western", "Rift Valley", "Rift Valley"]
})

region_lookup

,branch,region
0,Nairobi,Central
1,Mombasa,Coast
2,Kisumu,Western
3,Nakuru,Rift Valley
4,Eldoret,Rift Valley


In [35]:
import os
import pandas as pd
from openpyxl import load_workbook

os.makedirs("excel", exist_ok=True)

file_path = "excel/duka_branch_summary.xlsx"

# Create the workbook sheets
with pd.ExcelWriter(file_path, engine="openpyxl") as writer:
    sales.to_excel(writer, sheet_name="SalesData", index=False)
    branch_summary.to_excel(writer, sheet_name="BranchSummary", index=False)
    region_lookup.to_excel(writer, sheet_name="RegionLookup", index=False)
    sales.to_excel(writer, sheet_name="PivotSource", index=False)

print("Excel workbook created!")

Excel workbook created!


In [36]:
from openpyxl import load_workbook

wb = load_workbook("excel/duka_branch_summary.xlsx")

ws = wb["BranchSummary"]

# Add formula headings
ws["D1"] = "SUMIFS Revenue"
ws["E1"] = "Region (XLOOKUP)"

# Add formulas for all five branches
for row in range(2, 7):
    ws[f"D{row}"] = (
        f'=SUMIFS(SalesData!$H:$H,SalesData!$C:$C,A{row})'
    )

    ws[f"E{row}"] = (
        f'=XLOOKUP(A{row},RegionLookup!$A:$A,RegionLookup!$B:$B,"Not Found")'
    )

wb.save("excel/duka_branch_summary.xlsx")

print("SUMIFS and XLOOKUP formulas added!")

SUMIFS and XLOOKUP formulas added!


In [37]:
pivot_summary = pd.pivot_table(
    sales,
    values="revenue",
    index="branch",
    columns="channel",
    aggfunc="sum",
    fill_value=0,
    margins=True,
    margins_name="Grand Total"
)

pivot_summary

channel,In-store,Online,Phone order,Grand Total
branch,,,,
Eldoret,4885300,3450700,508200,8844200
Kisumu,6468600,3370800,1028700,10868100
Mombasa,8039600,4086800,1877300,14003700
Nairobi,14808400,7527400,3770300,26106100
Nakuru,6767300,4350400,852700,11970400
Grand Total,40969200,22786100,8037200,71792500


In [38]:
with pd.ExcelWriter(
    "excel/duka_branch_summary.xlsx",
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace"
) as writer:
    pivot_summary.to_excel(writer, sheet_name="PivotTable")

print("PivotTable summary added to Excel!")

PivotTable summary added to Excel!


In [39]:
import os

print(os.path.exists("excel/duka_branch_summary.xlsx"))
print(os.path.getsize("excel/duka_branch_summary.xlsx"), "bytes")

True
114370 bytes


In [40]:
from openpyxl import load_workbook

file_path = "excel/duka_branch_summary.xlsx"

wb = load_workbook(file_path, data_only=False)

print("Workbook sheets:", wb.sheetnames)
print("SUMIFS formula:", wb["BranchSummary"]["D2"].value)
print("XLOOKUP formula:", wb["BranchSummary"]["E2"].value)

print("Notebook revenue:", sales["revenue"].sum())
print("Branch summary revenue:", branch_summary["total_revenue"].sum())
print("PivotTable revenue:", pivot_summary.loc["Grand Total", "Grand Total"])

Workbook sheets: ['SalesData', 'BranchSummary', 'RegionLookup', 'PivotSource', 'PivotTable']
SUMIFS formula: =SUMIFS(SalesData!$H:$H,SalesData!$C:$C,A2)
XLOOKUP formula: =XLOOKUP(A2,RegionLookup!$A:$A,RegionLookup!$B:$B,"Not Found")
Notebook revenue: 71792500
Branch summary revenue: 71792500
PivotTable revenue: 71792500


In [41]:
from google.colab import files

files.download("excel/duka_branch_summary.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [42]:
import os

print(os.listdir())

['.config', 'agent_transactions.csv', 'outputs', 'generate_data.py', 'duka_sales.csv', 'excel', 'agents.csv', 'sample_data']


## Executive Summary

1. The dataset contains **1,000 sales orders** across five branches.
2. Total revenue from all branches was **KES 71,792,500**.
3. Nairobi was the best-performing branch, earning **KES 26,106,100**.
4. Laptops generated the highest revenue of **KES 41,535,000**.
5. April was the best month, earning **KES 13,518,300**.